# 04 — Análise do Corpus SciELO com Campos V3

Este notebook consolida os arquivos limpos gerados no passo 03, deduplica o corpus, revisa os campos de estudo com uma classificação V3 mais útil para análise e gera outputs técnicos e amigáveis para Google Sheets.

## O que este notebook faz

1. Encontra automaticamente a raiz do projeto.
2. Lê arquivos em `data/03_clean/`.
3. Ignora arquivos `_sheets.csv` na leitura técnica para evitar duplicidade.
4. Deduplica artigos por DOI ou título normalizado.
5. Cria colunas analíticas maximizadas:
   - `campo_principal_v3`
   - `campos_secundarios_v3`
   - `campos_estudo_v3`
   - `campo_scores_v3`
   - `campo_fontes_v3`
   - `aderencia_elites_v1`
6. Gera rankings, cruzamentos, Excel consolidado e amostra de 5% para validação manual.

In [ ]:
from pathlib import Path
from pandas.errors import EmptyDataError
from collections import Counter
from unidecode import unidecode
import ast
import html
import json
import os
import re

import numpy as np
import pandas as pd

## 1. Configuração de diretórios

A função abaixo sobe na árvore de pastas até encontrar a raiz do projeto. Isso permite rodar o notebook mesmo estando dentro de `Scielo/04.data_analysis/`.

In [ ]:
def find_project_dir(start=None):
    current = Path(start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        has_project_marker = (
            (candidate / 'README.md').exists()
            or (candidate / '.gitignore').exists()
            or (candidate / 'requirements.txt').exists()
        )
        has_expected_dirs = (
            (candidate / 'data').exists()
            or (candidate / 'Scielo').exists()
        )

        if has_project_marker and has_expected_dirs:
            return candidate

    return current


PROJECT_DIR = find_project_dir()

DATA_DIR = PROJECT_DIR / 'data'
CLEAN_DIR = DATA_DIR / '03_clean'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
ANALYSIS_DIR = OUTPUT_DIR / 'scielo_analysis'
LOG_DIR = PROJECT_DIR / 'logs'

for directory in [OUTPUT_DIR, ANALYSIS_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('CLEAN_DIR:', CLEAN_DIR)
print('ANALYSIS_DIR:', ANALYSIS_DIR)

## 2. Encontrar arquivos limpos

Padrões aceitos:

```text
03_corpus_*_scielo_clean.csv
*_scielo_clean.csv
*clean.csv
```

O notebook ignora os arquivos `_sheets.csv` na leitura técnica e, quando há arquivos por keyword, ignora o consolidado para evitar duplicidade.

In [ ]:
def find_clean_files(clean_dir):
    patterns = [
        '03_corpus_*_scielo_clean.csv',
        '*_scielo_clean.csv',
        '*clean.csv',
    ]

    files = []

    for pattern in patterns:
        files.extend(clean_dir.glob(pattern))

    files = sorted(set(files))

    # Remove versões amigáveis para Google Sheets para evitar duplicidade.
    files = [
        path for path in files
        if not path.name.endswith('_sheets.csv')
    ]

    # Se existirem arquivos por keyword, não lê o consolidado junto.
    keyword_files = [
        path for path in files
        if path.name != '03_corpus_scielo_clean_combined.csv'
    ]

    if keyword_files:
        return keyword_files

    return files


clean_files = find_clean_files(CLEAN_DIR)

print(f'Arquivos limpos encontrados: {len(clean_files)}')
for path in clean_files:
    size_kb = path.stat().st_size / 1024 if path.exists() else 0
    print(f'- {path} | {size_kb:.1f} KB')

In [ ]:
if len(clean_files) == 0:
    print('\n[DEBUG] Nenhum arquivo encontrado pelo padrão esperado.')
    print('Conteúdo de CLEAN_DIR:')

    if CLEAN_DIR.exists():
        for path in sorted(CLEAN_DIR.iterdir()):
            print('-', path.name)
    else:
        print(f'CLEAN_DIR não existe: {CLEAN_DIR}')

    raise FileNotFoundError(
        f'Nenhum arquivo limpo encontrado em {CLEAN_DIR}. '
        'Confira se o notebook 03 salvou os CSVs em data/03_clean.'
    )

## 3. Funções auxiliares

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ''

    value = str(value)
    value = html.unescape(value)
    value = re.sub(r'\s+', ' ', value).strip()

    return value


def normalize_text_key(value):
    value = clean_text(value)
    value = unidecode(value.lower())
    value = re.sub(r'[^a-z0-9]+', ' ', value).strip()

    return value


def safe_literal_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    value = str(value).strip()

    if value in ['', '[]', 'nan', 'None', '<NA>']:
        return []

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            return [clean_text(item) for item in parsed if clean_text(item)]

        return [clean_text(parsed)] if clean_text(parsed) else []

    except Exception:
        # Caso seja uma lista já transformada em string separada por pipe.
        if ' | ' in value:
            return [clean_text(item) for item in value.split(' | ') if clean_text(item)]

        return [clean_text(value)] if clean_text(value) else []


def list_to_sheets(value, sep=' | '):
    items = safe_literal_list(value)
    return sep.join([str(item).strip() for item in items if str(item).strip()])


def list_to_json(value):
    items = safe_literal_list(value)
    return json.dumps(items, ensure_ascii=False)


def dict_to_json(value):
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)

    if pd.isna(value):
        return '{}'

    return str(value)


def get_keyword_slug_from_clean_file(path):
    name = path.name
    name = name.replace('03_corpus_', '')
    name = name.replace('_scielo_clean.csv', '')
    name = name.replace('_scielo_clean_sheets.csv', '')

    return name

## 4. Leitura e padronização dos arquivos limpos

In [ ]:
def normalize_column_types(df):
    out = df.copy()

    list_like_cols = [
        'article_authors',
        'article_keywords',
        'article_references',
        'article_institutions',
        'institutions_raw_combined',
        'institutions_normalized',
        'institution_states',
        'institution_regions',
        'institution_countries',
        'states_detected',
        'regions_detected',
        'fields_detected',
        'methodologies_detected',
    ]

    for col in list_like_cols:
        if col in out.columns:
            out[col] = out[col].apply(safe_literal_list)

    text_cols = [
        'article_title',
        'article_journal',
        'article_doi',
        'article_language',
        'article_publisher',
        'article_abstract',
        'document_type_v2',
        'source_file',
        'keyword_slug',
        'search_keyword',
        'search_group',
        'main_institution',
        'main_state',
        'main_region',
        'main_country',
    ]

    for col in text_cols:
        if col in out.columns:
            out[col] = out[col].apply(clean_text)

    if 'article_year' in out.columns:
        out['article_year'] = pd.to_numeric(out['article_year'], errors='coerce').astype('Int64')

    if 'article_references_count' in out.columns:
        out['article_references_count'] = pd.to_numeric(
            out['article_references_count'],
            errors='coerce'
        ).fillna(0).astype(int)

    return out


def read_clean_file(path):
    if not path.exists():
        return None, {'file': str(path), 'reason': 'file does not exist'}

    if path.stat().st_size == 0:
        return None, {'file': str(path), 'reason': 'empty file'}

    try:
        df = pd.read_csv(path)
    except EmptyDataError:
        return None, {'file': str(path), 'reason': 'empty csv / no columns'}
    except Exception as exc:
        return None, {'file': str(path), 'reason': str(exc)}

    if df.empty:
        return None, {'file': str(path), 'reason': 'no rows'}

    df['source_file'] = path.name

    if 'keyword_slug' not in df.columns:
        df['keyword_slug'] = get_keyword_slug_from_clean_file(path)

    df = normalize_column_types(df)

    return df, None


loaded = []
skipped = []

for path in clean_files:
    df, error = read_clean_file(path)

    if error:
        skipped.append(error)
        print(f'[SKIP] {path.name}: {error["reason"]}')
        continue

    loaded.append(df)
    print(f'[OK] {path.name}: {len(df)} linhas')

if skipped:
    skipped_df = pd.DataFrame(skipped)
    skipped_df.to_csv(LOG_DIR / 'skipped_clean_files_analysis.csv', index=False, encoding='utf-8-sig')

if not loaded:
    raise RuntimeError('Nenhum arquivo limpo válido foi carregado.')

corpus_repeated = pd.concat(loaded, ignore_index=True)

print('\nCorpus com repetição por termo:', corpus_repeated.shape)
corpus_repeated.head()

## 5. Deduplicação

Deduplicação por DOI quando disponível; caso contrário, por título normalizado.

In [ ]:
def make_dedup_key(row):
    doi = clean_text(row.get('article_doi', '')).lower()

    if doi:
        return 'doi::' + doi

    title = normalize_text_key(row.get('article_title', ''))

    return 'title::' + title


corpus_repeated['dedup_key'] = corpus_repeated.apply(make_dedup_key, axis=1)

corpus_unique = (
    corpus_repeated
    .sort_values(
        by=['article_doi', 'article_title'],
        na_position='last'
    )
    .drop_duplicates(subset=['dedup_key'], keep='first')
    .copy()
)

print('Corpus com repetição:', corpus_repeated.shape)
print('Corpus único:', corpus_unique.shape)

## 6. Classificador V3 de campo de estudo

A V3 procura evitar superclassificação. Ela separa:

- campo principal;
- campos secundários;
- score por campo;
- fonte da classificação.

A lógica usa pesos diferentes para periódico, título, palavras-chave e resumo.

In [ ]:
FIELD_RULES_V3 = {
    'Sociologia': {
        'journals': [
            'tempo social',
            'revista brasileira de ciencias sociais',
            'sociologias',
            'sociedade e estado',
            'caderno crh',
            'dados',
            'revista de sociologia e politica',
        ],
        'strong': [
            'sociologia das elites',
            'sociologia',
            'bourdieu',
            'campo do poder',
            'capital simbolico',
            'campo social',
            'campo politico',
            'classes dirigentes',
            'classe dominante',
            'classes dominantes',
            'elites sociais',
            'elite social',
            'estratificacao social',
        ],
        'weak': [
            'desigualdade',
            'distincao',
            'habitus',
            'trajetoria social',
            'capital cultural',
        ],
    },

    'Ciência Política': {
        'journals': [
            'revista de sociologia e politica',
            'opinia o publica',
            'opiniao publica',
            'revista brasileira de ciencia politica',
            'lua nova',
            'brazilian political science review',
        ],
        'strong': [
            'elite politica',
            'elites politicas',
            'classe politica',
            'parlamento',
            'camara dos deputados',
            'senado',
            'partidos politicos',
            'representacao politica',
            'eleicoes',
            'democracia',
            'quem governa',
            'poder politico',
        ],
        'weak': [
            'estado',
            'governo',
            'politica publica',
            'voto',
            'campanha eleitoral',
        ],
    },

    'História': {
        'journals': [
            'historia',
            'revista brasileira de historia',
            'topoi',
            'almanack',
            'varia historia',
            'estudos historicos',
        ],
        'strong': [
            'historia',
            'historiografia',
            'seculo xix',
            'seculo xx',
            'imperio',
            'primeira republica',
            'republica velha',
            'arquivo historico',
            'fontes historicas',
        ],
        'weak': [
            'memoria',
            'trajetoria historica',
            'periodo colonial',
            'monarquia',
        ],
    },

    'Direito': {
        'journals': [
            'direito',
            'revista direito gv',
            'direito e praxis',
        ],
        'strong': [
            'direito',
            'juridico',
            'judiciario',
            'magistratura',
            'supremo tribunal',
            'stf',
            'tribunal',
            'constituicao',
            'advocacia',
        ],
        'weak': [
            'justica',
            'norma juridica',
            'legal',
            'legislacao',
        ],
    },

    'Educação': {
        'journals': [
            'educacao e pesquisa',
            'educacao & sociedade',
            'cadernos de pesquisa',
            'revista brasileira de educacao',
            'pro-posicoes',
        ],
        'strong': [
            'educacao',
            'ensino superior',
            'universidade',
            'escola',
            'sistema educacional',
            'professores',
            'capital escolar',
            'trajetoria escolar',
        ],
        'weak': [
            'aprendizagem',
            'curriculo',
            'formacao docente',
            'estudantes',
        ],
    },

    'Economia': {
        'journals': [
            'economia',
            'nova economia',
            'economia e sociedade',
            'revista de economia politica',
        ],
        'strong': [
            'economia',
            'economico',
            'elite economica',
            'elites economicas',
            'burguesia',
            'burguesia industrial',
            'burguesia financeira',
            'burguesia bancaria',
            'mercado financeiro',
            'capital economico',
            'empresariado',
        ],
        'weak': [
            'renda',
            'capitalismo',
            'industrializacao',
            'empresa',
            'financeiro',
        ],
    },

    'Administração/Gestão Pública': {
        'journals': [
            'revista de administracao publica',
            'cadernos ebape',
            'organizacoes & sociedade',
            'rae',
        ],
        'strong': [
            'administracao publica',
            'gestao publica',
            'burocracia',
            'elite burocratica',
            'tecnocracia',
            'governanca',
            'gestores publicos',
        ],
        'weak': [
            'organizacoes',
            'gestao',
            'administracao',
            'planejamento',
        ],
    },

    'Geografia': {
        'journals': [
            'revista brasileira de geografia',
            'geousp',
            'mercator',
            'territorio',
        ],
        'strong': [
            'geografia',
            'territorio',
            'espaco urbano',
            'segregacao espacial',
            'regiao metropolitana',
            'urbanizacao',
        ],
        'weak': [
            'cidade',
            'regional',
            'espacial',
            'territorial',
        ],
    },

    'Antropologia': {
        'journals': [
            'mana',
            'revista de antropologia',
            'vibrant',
            'horizontes antropologicos',
        ],
        'strong': [
            'antropologia',
            'etnografia',
            'trabalho de campo',
            'parentesco',
            'ritual',
        ],
        'weak': [
            'cultura',
            'simbolico',
            'identidade',
            'praticas culturais',
        ],
    },

    'Comunicação': {
        'journals': [
            'comunicacao',
            'galaxia',
            'matrizes',
            'intercom',
        ],
        'strong': [
            'comunicacao',
            'midia',
            'jornalismo',
            'imprensa',
            'opiniao publica digital',
            'redes sociais',
            'twitter',
        ],
        'weak': [
            'discurso midiatico',
            'esfera publica digital',
            'plataformas digitais',
        ],
    },

    'Relações Internacionais': {
        'journals': [
            'contexto internacional',
            'revista brasileira de politica internacional',
        ],
        'strong': [
            'relacoes internacionais',
            'politica externa',
            'diplomacia',
            'sistema internacional',
            'elite diplomatica',
        ],
        'weak': [
            'internacional',
            'global',
            'geopolitica',
        ],
    },

    'Psicologia': {
        'journals': [
            'psicologia',
            'estudos de psicologia',
            'psicologia reflexao e critica',
        ],
        'strong': [
            'psicologia',
            'psicologico',
            'subjetividade',
            'saude mental',
        ],
        'weak': [
            'comportamento',
            'identidade',
            'percepcao',
        ],
    },

    'Saúde': {
        'journals': [
            'cadernos de saude publica',
            'saude publica',
            'ciencia & saude coletiva',
            'historia ciencias saude',
            'revista de saude publica',
            'saude e sociedade',
        ],
        'strong': [
            'saude',
            'medicina',
            'medico',
            'hospital',
            'epidemiologia',
            'saude publica',
            'sistema de saude',
            'doenca',
            'pacientes',
        ],
        'weak': [
            'clinico',
            'sanitario',
            'mortalidade',
            'prevalencia',
            'enfermagem',
        ],
    },

    'Ciências Agrárias': {
        'journals': [
            'pesquisa agropecuaria',
            'bragantia',
            'scientia agricola',
            'ciencia rural',
        ],
        'strong': [
            'agricultura',
            'agronomia',
            'soja',
            'milho',
            'cultivar',
            'sementes',
            'elite lines',
            'linhagens elite',
        ],
        'weak': [
            'solo',
            'plantas',
            'produtividade agricola',
            'pecuaria',
        ],
    },

    'Ciências Biológicas': {
        'journals': [
            'biologia',
            'genetics and molecular biology',
            'brazilian journal of biology',
        ],
        'strong': [
            'biologia',
            'genetica',
            'especie dominante',
            'dominant species',
            'biodiversidade',
            'ecologia',
        ],
        'weak': [
            'animal',
            'vegetal',
            'molecular',
            'populacao biologica',
        ],
    },

    'Ciências Exatas': {
        'journals': [
            'matematica',
            'fisica',
            'quimica',
            'engenharia',
        ],
        'strong': [
            'matematica',
            'fisica',
            'quimica',
            'engenharia',
            'algoritmo',
            'modelo computacional',
        ],
        'weak': [
            'simulacao',
            'otimizacao',
            'metodo numerico',
        ],
    },

    'Filosofia': {
        'journals': [
            'filosofia',
            'kriterion',
            'trans/form/acao',
        ],
        'strong': [
            'filosofia',
            'filosofico',
            'teoria critica',
            'etica',
            'epistemologia',
        ],
        'weak': [
            'pensamento',
            'conceito',
            'ontologia',
        ],
    },

    'Linguística/Letras': {
        'journals': [
            'letras',
            'linguistica',
            'delta',
            'trabalhos em linguistica aplicada',
        ],
        'strong': [
            'linguistica',
            'literatura',
            'analise literaria',
            'letras',
            'discurso',
        ],
        'weak': [
            'narrativa',
            'romance',
            'texto',
            'linguagem',
        ],
    },

    'Estudos de Gênero/Sexualidade': {
        'journals': [
            'cadernos pagu',
            'revista estudos feministas',
        ],
        'strong': [
            'genero',
            'feminismo',
            'mulheres',
            'sexualidade',
            'lgbt',
            'masculinidade',
        ],
        'weak': [
            'corpo',
            'familia',
            'relações de genero',
        ],
    },

    'Serviço Social': {
        'journals': [
            'servico social',
            'katálysis',
            'katalysis',
        ],
        'strong': [
            'servico social',
            'assistencia social',
            'politica social',
            'questao social',
        ],
        'weak': [
            'vulnerabilidade social',
            'proteção social',
        ],
    },

    'Educação Física/Esporte': {
        'journals': [
            'educacao fisica',
            'revista brasileira de ciencias do esporte',
        ],
        'strong': [
            'educacao fisica',
            'esporte',
            'atletas',
            'elite athletes',
            'atletas de elite',
        ],
        'weak': [
            'performance esportiva',
            'treinamento',
        ],
    },

    'Ciência da Informação': {
        'journals': [
            'ciencia da informacao',
            'perspectivas em ciencia da informacao',
        ],
        'strong': [
            'ciencia da informacao',
            'bibliometria',
            'cientometria',
            'producao cientifica',
            'bases de dados',
        ],
        'weak': [
            'informacao',
            'periodicos cientificos',
            'indexacao',
        ],
    },
}


FIELD_PRIORITY = [
    'Sociologia',
    'Ciência Política',
    'História',
    'Educação',
    'Economia',
    'Direito',
    'Administração/Gestão Pública',
    'Antropologia',
    'Geografia',
    'Comunicação',
    'Relações Internacionais',
    'Estudos de Gênero/Sexualidade',
    'Filosofia',
    'Serviço Social',
    'Ciência da Informação',
    'Psicologia',
    'Saúde',
    'Ciências Agrárias',
    'Ciências Biológicas',
    'Ciências Exatas',
    'Linguística/Letras',
    'Educação Física/Esporte',
]


def score_field_v3(row, field, rules):
    journal = normalize_text_key(row.get('article_journal', ''))
    title = normalize_text_key(row.get('article_title', ''))
    abstract = normalize_text_key(row.get('article_abstract', ''))
    keywords = normalize_text_key(list_to_sheets(row.get('article_keywords', [])))

    score = 0
    sources = []

    for term in rules.get('journals', []):
        term_key = normalize_text_key(term)
        if term_key and term_key in journal:
            score += 6
            sources.append(f'periodico:{term}')
            break

    for term in rules.get('strong', []):
        term_key = normalize_text_key(term)

        if not term_key:
            continue

        if term_key in title:
            score += 5
            sources.append(f'titulo:{term}')

        if term_key in keywords:
            score += 4
            sources.append(f'palavra_chave:{term}')

        if term_key in abstract:
            score += 2
            sources.append(f'resumo:{term}')

    for term in rules.get('weak', []):
        term_key = normalize_text_key(term)

        if not term_key:
            continue

        if term_key in title:
            score += 2
            sources.append(f'titulo_fraco:{term}')

        if term_key in keywords:
            score += 2
            sources.append(f'palavra_chave_fraca:{term}')

        if term_key in abstract:
            score += 1
            sources.append(f'resumo_fraco:{term}')

    return score, sources


def classify_fields_v3(row):
    field_scores = {}
    field_sources = {}

    for field, rules in FIELD_RULES_V3.items():
        score, sources = score_field_v3(row, field, rules)

        if score > 0:
            field_scores[field] = score
            field_sources[field] = sources

    if not field_scores:
        return pd.Series({
            'campo_principal_v3': 'Indefinido',
            'campos_secundarios_v3': [],
            'campos_estudo_v3': ['Indefinido'],
            'campo_scores_v3': {},
            'campo_fontes_v3': {},
        })

    priority_index = {field: idx for idx, field in enumerate(FIELD_PRIORITY)}

    sorted_fields = sorted(
        field_scores.keys(),
        key=lambda field: (
            -field_scores[field],
            priority_index.get(field, 999),
            field
        )
    )

    main_field = sorted_fields[0]
    main_score = field_scores[main_field]

    secondary_fields = [
        field
        for field in sorted_fields[1:]
        if field_scores[field] >= max(3, main_score * 0.45)
    ][:2]

    fields_final = [main_field] + secondary_fields

    return pd.Series({
        'campo_principal_v3': main_field,
        'campos_secundarios_v3': secondary_fields,
        'campos_estudo_v3': fields_final,
        'campo_scores_v3': field_scores,
        'campo_fontes_v3': field_sources,
    })


field_rows = corpus_unique.apply(classify_fields_v3, axis=1)
corpus_unique = pd.concat([corpus_unique.reset_index(drop=True), field_rows.reset_index(drop=True)], axis=1)

# Também aplica na base repetida para análises por termo.
field_rows_repeated = corpus_repeated.apply(classify_fields_v3, axis=1)
corpus_repeated = pd.concat([corpus_repeated.reset_index(drop=True), field_rows_repeated.reset_index(drop=True)], axis=1)

corpus_unique[['article_title', 'article_journal', 'campo_principal_v3', 'campos_secundarios_v3', 'campo_scores_v3']].head()

## 7. Aderência temática ao campo de elites

Esta classificação separa o problema disciplinar do problema temático. Um artigo pode ser de Saúde, Agrárias ou Biológicas, mas ter baixa aderência ao campo sociológico/político das elites.

Categorias:

- `Alta`
- `Média`
- `Baixa`
- `Fora do escopo provável`

In [ ]:
ELITES_HIGH_TERMS = [
    'sociologia das elites',
    'elite politica',
    'elites politicas',
    'elite economica',
    'elites economicas',
    'elite empresarial',
    'elite financeira',
    'classes dirigentes',
    'classe dirigente',
    'grupo dirigente',
    'grupos dirigentes',
    'classe dominante',
    'classes dominantes',
    'campo do poder',
    'grupo de poder',
    'quem governa',
    'burguesia industrial',
    'burguesia financeira',
    'burguesia bancaria',
]

ELITES_MEDIUM_TERMS = [
    'elites',
    'elite',
    'burguesia',
    'poder politico',
    'poder economico',
    'dominacao social',
    'capital simbolico',
    'capital economico',
    'capital cultural',
    'estratificacao',
]

NOISE_TERMS = [
    'elite athletes',
    'atletas de elite',
    'elite lines',
    'linhagens elite',
    'dominant species',
    'especie dominante',
    'power analysis',
    'statistical power',
    'poder calorifico',
]


def classify_elites_adherence(row):
    text = normalize_text_key(' '.join([
        str(row.get('article_title', '')),
        str(row.get('article_abstract', '')),
        list_to_sheets(row.get('article_keywords', [])),
        str(row.get('keyword_slug', '')),
        str(row.get('search_keyword', '')),
    ]))

    high_hits = [term for term in ELITES_HIGH_TERMS if normalize_text_key(term) in text]
    medium_hits = [term for term in ELITES_MEDIUM_TERMS if normalize_text_key(term) in text]
    noise_hits = [term for term in NOISE_TERMS if normalize_text_key(term) in text]

    main_field = row.get('campo_principal_v3', '')

    if high_hits:
        adherence = 'Alta'
    elif len(medium_hits) >= 2:
        adherence = 'Média'
    elif medium_hits and main_field in [
        'Sociologia',
        'Ciência Política',
        'História',
        'Economia',
        'Direito',
        'Educação',
        'Administração/Gestão Pública',
        'Antropologia',
        'Geografia',
        'Comunicação',
        'Relações Internacionais',
    ]:
        adherence = 'Média'
    elif medium_hits:
        adherence = 'Baixa'
    else:
        adherence = 'Fora do escopo provável'

    if noise_hits and not high_hits:
        if adherence == 'Média':
            adherence = 'Baixa'
        elif adherence == 'Baixa':
            adherence = 'Fora do escopo provável'

    return pd.Series({
        'aderencia_elites_v1': adherence,
        'termos_aderencia_alta': high_hits,
        'termos_aderencia_media': medium_hits,
        'termos_ruido_provavel': noise_hits,
    })


adherence_rows = corpus_unique.apply(classify_elites_adherence, axis=1)
corpus_unique = pd.concat([corpus_unique.reset_index(drop=True), adherence_rows.reset_index(drop=True)], axis=1)

adherence_rows_repeated = corpus_repeated.apply(classify_elites_adherence, axis=1)
corpus_repeated = pd.concat([corpus_repeated.reset_index(drop=True), adherence_rows_repeated.reset_index(drop=True)], axis=1)

corpus_unique['aderencia_elites_v1'].value_counts(dropna=False)

## 8. Diagnóstico geral

In [ ]:
def has_list_values(series):
    return series.apply(lambda x: isinstance(x, list) and len(x) > 0).sum()


diagnostic = pd.DataFrame({
    'metric': [
        'arquivos_lidos',
        'linhas_com_repeticao_por_termo',
        'linhas_unicas_deduplicadas',
        'titulos_unicos',
        'dois_unicos',
        'periodicos_unicos',
        'autores_unicos',
        'registros_com_resumo',
        'registros_com_palavras_chave',
        'registros_com_instituicoes',
        'registros_com_estados',
        'registros_com_regioes',
        'ano_minimo',
        'ano_maximo',
        'aderencia_alta',
        'aderencia_media',
        'aderencia_baixa',
        'fora_escopo_provavel',
    ],
    'value': [
        len(loaded),
        len(corpus_repeated),
        len(corpus_unique),
        corpus_unique['article_title'].nunique() if 'article_title' in corpus_unique.columns else 0,
        corpus_unique['article_doi'].replace('', pd.NA).dropna().nunique() if 'article_doi' in corpus_unique.columns else 0,
        corpus_unique['article_journal'].replace('', pd.NA).dropna().nunique() if 'article_journal' in corpus_unique.columns else 0,
        corpus_unique['article_authors'].explode().dropna().nunique() if 'article_authors' in corpus_unique.columns else 0,
        corpus_unique['article_abstract'].replace('', pd.NA).notna().sum() if 'article_abstract' in corpus_unique.columns else 0,
        has_list_values(corpus_unique['article_keywords']) if 'article_keywords' in corpus_unique.columns else 0,
        has_list_values(corpus_unique['institutions_normalized']) if 'institutions_normalized' in corpus_unique.columns else 0,
        has_list_values(corpus_unique['institution_states']) if 'institution_states' in corpus_unique.columns else 0,
        has_list_values(corpus_unique['institution_regions']) if 'institution_regions' in corpus_unique.columns else 0,
        corpus_unique['article_year'].min() if 'article_year' in corpus_unique.columns else pd.NA,
        corpus_unique['article_year'].max() if 'article_year' in corpus_unique.columns else pd.NA,
        (corpus_unique['aderencia_elites_v1'] == 'Alta').sum(),
        (corpus_unique['aderencia_elites_v1'] == 'Média').sum(),
        (corpus_unique['aderencia_elites_v1'] == 'Baixa').sum(),
        (corpus_unique['aderencia_elites_v1'] == 'Fora do escopo provável').sum(),
    ]
})

diagnostic

## 9. Rankings principais

In [ ]:
def rank_exploded(df, list_col, output_col, title_col='article_title'):
    if list_col not in df.columns:
        return pd.DataFrame(columns=[output_col, 'productions'])

    return (
        df[[title_col, list_col]]
        .explode(list_col)
        .dropna(subset=[list_col])
        .assign(**{output_col: lambda x: x[list_col].astype(str).str.strip()})
        .query(f'{output_col} != ""')
        .groupby(output_col, as_index=False)
        .agg(productions=(title_col, 'count'))
        .sort_values('productions', ascending=False)
    )


rank_authors = rank_exploded(corpus_unique, 'article_authors', 'author')
rank_keywords = rank_exploded(corpus_unique, 'article_keywords', 'keyword')
rank_institutions = rank_exploded(corpus_unique, 'institutions_normalized', 'institution')
rank_states = rank_exploded(corpus_unique, 'institution_states', 'state')
rank_regions = rank_exploded(corpus_unique, 'institution_regions', 'region')
rank_fields_v3 = rank_exploded(corpus_unique, 'campos_estudo_v3', 'field')
rank_fields_main_v3 = (
    corpus_unique
    .groupby('campo_principal_v3', as_index=False)
    .agg(productions=('article_title', 'count'))
    .sort_values('productions', ascending=False)
)
rank_adherence = (
    corpus_unique
    .groupby('aderencia_elites_v1', as_index=False)
    .agg(productions=('article_title', 'count'))
    .sort_values('productions', ascending=False)
)

rank_journals = (
    corpus_unique
    .assign(journal=lambda x: x['article_journal'].astype(str).str.strip() if 'article_journal' in x.columns else '')
    .query('journal != ""')
    .groupby('journal', as_index=False)
    .agg(
        productions=('article_title', 'count'),
        first_year=('article_year', 'min'),
        last_year=('article_year', 'max')
    )
    .sort_values('productions', ascending=False)
)

rank_years = (
    corpus_unique
    .dropna(subset=['article_year'])
    .groupby('article_year', as_index=False)
    .agg(productions=('article_title', 'count'))
    .sort_values('article_year')
)

rank_fields_main_v3.head(20)

## 10. Metodologias

Se a coluna já veio do passo 03, ela será usada. Caso contrário, gera uma V1 por heurística.

In [ ]:
METHODOLOGY_PATTERNS = {
    'Quantitativa': [
        'quantitativo', 'estatistica', 'regressao', 'modelo', 'survey',
        'questionario', 'base de dados', 'banco de dados', 'amostra',
        'analise fatorial', 'analise de correspondencia', 'cluster',
        'serie temporal', 'dados longitudinais'
    ],
    'Qualitativa': [
        'qualitativo', 'entrevista', 'entrevistas', 'etnografia',
        'observacao participante', 'estudo de caso', 'historia oral',
        'analise de discurso', 'trajetoria', 'narrativa'
    ],
    'Bibliográfica/Teórica': [
        'revisao bibliografica', 'revisao de literatura', 'ensaio teorico',
        'discussao teorica', 'estado da arte', 'literatura', 'teoria'
    ],
    'Documental': [
        'documental', 'arquivo', 'fontes documentais', 'documentos',
        'atas', 'relatorios', 'legislacao', 'jornais'
    ],
    'Comparativa': [
        'comparativo', 'comparativa', 'comparacao', 'comparada',
        'entre paises', 'entre regioes', 'cross-national'
    ],
}


def classify_methodology(row):
    if 'methodologies_detected' in row.index:
        existing = safe_literal_list(row.get('methodologies_detected', []))
        if existing:
            return existing

    text = normalize_text_key(' '.join([
        str(row.get('article_title', '')),
        str(row.get('article_abstract', '')),
        str(row.get('article_full_text_sample', ''))[:12000],
    ]))

    detected = []

    for method, terms in METHODOLOGY_PATTERNS.items():
        if any(normalize_text_key(term) in text for term in terms):
            detected.append(method)

    if 'Quantitativa' in detected and 'Qualitativa' in detected:
        return ['Mista'] + detected

    return detected if detected else ['Não identificada']


corpus_unique['methodologies_detected_final'] = corpus_unique.apply(classify_methodology, axis=1)
corpus_repeated['methodologies_detected_final'] = corpus_repeated.apply(classify_methodology, axis=1)

rank_methodologies = rank_exploded(corpus_unique, 'methodologies_detected_final', 'methodology')

rank_methodologies

## 11. Autores citados V1

Parsing heurístico das referências bibliográficas. Esta saída exige validação posterior.

In [ ]:
def parse_reference_author_candidates(reference):
    reference = clean_text(reference)

    if not reference:
        return []

    candidates = []

    # SOBRENOME, Nome
    upper_matches = re.findall(
        r'\b([A-ZÁÉÍÓÚÂÊÔÃÕÇ]{3,}(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ]{3,})?),\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇA-Za-záéíóúâêôãõç]',
        reference
    )

    # Sobrenome, Nome
    title_matches = re.findall(
        r'\b([A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+),\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+',
        reference
    )

    candidates.extend(upper_matches)
    candidates.extend(title_matches)

    blacklist = {
        'revista', 'volume', 'editora', 'universidade', 'press',
        'janeiro', 'paulo', 'rio', 'brasil', 'https', 'doi',
        'scielo', 'editorial', 'dados'
    }

    cleaned = []

    for candidate in candidates:
        candidate_norm = clean_text(candidate).title()
        candidate_key = normalize_text_key(candidate_norm)

        if candidate_key not in blacklist and len(candidate_norm) > 2:
            cleaned.append(candidate_norm)

    return list(dict.fromkeys(cleaned))


if 'article_references' in corpus_unique.columns:
    references_long = (
        corpus_unique[['article_title', 'article_references']]
        .explode('article_references')
        .dropna(subset=['article_references'])
        .assign(reference=lambda x: x['article_references'].astype(str))
    )

    references_long['cited_author_candidates'] = references_long['reference'].apply(parse_reference_author_candidates)

    rank_cited_authors = (
        references_long[['article_title', 'cited_author_candidates']]
        .explode('cited_author_candidates')
        .dropna(subset=['cited_author_candidates'])
        .groupby('cited_author_candidates', as_index=False)
        .agg(
            citations_in_references=('article_title', 'count'),
            articles_citing=('article_title', 'nunique')
        )
        .sort_values(['articles_citing', 'citations_in_references'], ascending=False)
    )
else:
    references_long = pd.DataFrame()
    rank_cited_authors = pd.DataFrame(columns=['cited_author_candidates', 'citations_in_references', 'articles_citing'])

rank_cited_authors.head(30)

## 12. Cruzamentos analíticos

In [ ]:
field_year = (
    corpus_unique[['article_title', 'article_year', 'campos_estudo_v3']]
    .explode('campos_estudo_v3')
    .dropna(subset=['article_year'])
    .groupby(['article_year', 'campos_estudo_v3'], as_index=False)
    .agg(productions=('article_title', 'count'))
    .sort_values(['article_year', 'productions'], ascending=[True, False])
)

field_main_year = (
    corpus_unique
    .dropna(subset=['article_year'])
    .groupby(['article_year', 'campo_principal_v3'], as_index=False)
    .agg(productions=('article_title', 'count'))
    .sort_values(['article_year', 'productions'], ascending=[True, False])
)

method_year = (
    corpus_unique[['article_title', 'article_year', 'methodologies_detected_final']]
    .explode('methodologies_detected_final')
    .dropna(subset=['article_year'])
    .groupby(['article_year', 'methodologies_detected_final'], as_index=False)
    .agg(productions=('article_title', 'count'))
    .sort_values(['article_year', 'productions'], ascending=[True, False])
)

region_year = (
    corpus_unique[['article_title', 'article_year', 'institution_regions']]
    .explode('institution_regions')
    .dropna(subset=['article_year', 'institution_regions'])
    .groupby(['article_year', 'institution_regions'], as_index=False)
    .agg(productions=('article_title', 'count'))
    .sort_values(['article_year', 'productions'], ascending=[True, False])
)

field_region = (
    corpus_unique[['article_title', 'campo_principal_v3', 'institution_regions']]
    .explode('institution_regions')
    .dropna(subset=['campo_principal_v3', 'institution_regions'])
    .groupby(['campo_principal_v3', 'institution_regions'], as_index=False)
    .agg(productions=('article_title', 'count'))
    .sort_values('productions', ascending=False)
)

adherence_by_field = (
    corpus_unique
    .groupby(['campo_principal_v3', 'aderencia_elites_v1'], as_index=False)
    .agg(productions=('article_title', 'count'))
    .sort_values(['campo_principal_v3', 'productions'], ascending=[True, False])
)

adherence_by_keyword = (
    corpus_repeated
    .groupby(['keyword_slug', 'aderencia_elites_v1'], as_index=False)
    .agg(productions=('article_title', 'count'))
    .sort_values(['keyword_slug', 'productions'], ascending=[True, False])
)

## 13. Bases amigáveis para Google Sheets

In [ ]:
def make_sheets_df(df):
    out = pd.DataFrame()

    def col(name, default=''):
        return df[name] if name in df.columns else default

    out['termo_busca'] = col('search_keyword', col('keyword_slug', ''))
    out['slug_termo_busca'] = col('keyword_slug', '')
    out['grupo_termo_busca'] = col('search_group', '')
    out['titulo'] = col('article_title', '')
    out['autores'] = df['article_authors'].apply(list_to_sheets) if 'article_authors' in df.columns else ''
    out['periodico'] = col('article_journal', '')
    out['ano'] = col('article_year', pd.NA)
    out['doi'] = col('article_doi', '')
    out['idioma'] = col('article_language', '')
    out['resumo'] = col('article_abstract', '')
    out['palavras_chave'] = df['article_keywords'].apply(list_to_sheets) if 'article_keywords' in df.columns else ''
    out['instituicoes_normalizadas'] = df['institutions_normalized'].apply(list_to_sheets) if 'institutions_normalized' in df.columns else ''
    out['instituicao_principal'] = col('main_institution', '')
    out['estados'] = df['institution_states'].apply(list_to_sheets) if 'institution_states' in df.columns else ''
    out['regioes'] = df['institution_regions'].apply(list_to_sheets) if 'institution_regions' in df.columns else ''
    out['paises'] = df['institution_countries'].apply(list_to_sheets) if 'institution_countries' in df.columns else ''
    out['estado_principal'] = col('main_state', '')
    out['regiao_principal'] = col('main_region', '')
    out['pais_principal'] = col('main_country', '')

    out['campo_principal_v3'] = col('campo_principal_v3', '')
    out['campos_secundarios_v3'] = df['campos_secundarios_v3'].apply(list_to_sheets) if 'campos_secundarios_v3' in df.columns else ''
    out['campos_estudo_v3'] = df['campos_estudo_v3'].apply(list_to_sheets) if 'campos_estudo_v3' in df.columns else ''
    out['campo_scores_v3'] = df['campo_scores_v3'].apply(dict_to_json) if 'campo_scores_v3' in df.columns else ''
    out['campo_fontes_v3'] = df['campo_fontes_v3'].apply(dict_to_json) if 'campo_fontes_v3' in df.columns else ''

    out['aderencia_elites_v1'] = col('aderencia_elites_v1', '')
    out['termos_aderencia_alta'] = df['termos_aderencia_alta'].apply(list_to_sheets) if 'termos_aderencia_alta' in df.columns else ''
    out['termos_aderencia_media'] = df['termos_aderencia_media'].apply(list_to_sheets) if 'termos_aderencia_media' in df.columns else ''
    out['termos_ruido_provavel'] = df['termos_ruido_provavel'].apply(list_to_sheets) if 'termos_ruido_provavel' in df.columns else ''

    out['metodologias'] = df['methodologies_detected_final'].apply(list_to_sheets) if 'methodologies_detected_final' in df.columns else ''
    out['tipo_documento'] = col('document_type_v2', '')
    out['eh_artigo_pesquisa'] = col('is_core_research_article_v2', '')
    out['qtd_referencias'] = col('article_references_count', 0)
    out['arquivo_origem'] = col('source_file', '')
    out['chave_deduplicacao'] = col('dedup_key', '')

    return out


corpus_repeated_sheets = make_sheets_df(corpus_repeated)
corpus_unique_sheets = make_sheets_df(corpus_unique)

corpus_unique_sheets.head()

## 14. Amostra para validação manual de 5%

A amostra deve ser validada manualmente pela leitura integral do resumo.

In [ ]:
VALIDATION_SAMPLE_RATE = 0.05
RANDOM_STATE = 42

n_validation = max(1, int(np.ceil(len(corpus_unique_sheets) * VALIDATION_SAMPLE_RATE)))

manual_validation_sample = (
    corpus_unique_sheets
    .sample(n=n_validation, random_state=RANDOM_STATE)
    .copy()
)

manual_validation_sample['validacao_manual_aderente'] = ''
manual_validation_sample['validacao_manual_campo_correto'] = ''
manual_validation_sample['validacao_manual_observacao'] = ''

print(f'Amostra de validação: {n_validation} linhas de {len(corpus_unique_sheets)} ({VALIDATION_SAMPLE_RATE:.0%})')
manual_validation_sample.head()

## 15. Salvar outputs

In [ ]:
# Bases técnicas
corpus_repeated.to_csv(
    ANALYSIS_DIR / 'corpus_elites_scielo_multitermos_with_term_repetition.csv',
    index=False,
    encoding='utf-8-sig'
)

corpus_unique.to_csv(
    ANALYSIS_DIR / 'corpus_elites_scielo_multitermos_analytical_unique.csv',
    index=False,
    encoding='utf-8-sig'
)

# Bases amigáveis para Google Sheets
corpus_repeated_sheets.to_csv(
    ANALYSIS_DIR / 'corpus_elites_scielo_multitermos_with_term_repetition_sheets.csv',
    index=False,
    encoding='utf-8-sig'
)

corpus_unique_sheets.to_csv(
    ANALYSIS_DIR / 'corpus_elites_scielo_multitermos_analytical_unique_sheets.csv',
    index=False,
    encoding='utf-8-sig'
)

# Nome específico para a versão com campos V3
corpus_unique_sheets.to_csv(
    ANALYSIS_DIR / 'corpus_elites_scielo_multitermos_analytical_unique_sheets_campos_v3.csv',
    index=False,
    encoding='utf-8-sig'
)

manual_validation_sample.to_csv(
    ANALYSIS_DIR / 'amostra_validacao_manual_5pct_sheets.csv',
    index=False,
    encoding='utf-8-sig'
)

# Diagnóstico
diagnostic.to_csv(ANALYSIS_DIR / 'diagnostic_analysis.csv', index=False, encoding='utf-8-sig')

# Rankings
rank_authors.to_csv(ANALYSIS_DIR / 'rank_01_autores_produtores.csv', index=False, encoding='utf-8-sig')
rank_journals.to_csv(ANALYSIS_DIR / 'rank_02_periodicos.csv', index=False, encoding='utf-8-sig')
rank_keywords.to_csv(ANALYSIS_DIR / 'rank_03_palavras_chave.csv', index=False, encoding='utf-8-sig')
rank_institutions.to_csv(ANALYSIS_DIR / 'rank_04_instituicoes.csv', index=False, encoding='utf-8-sig')
rank_states.to_csv(ANALYSIS_DIR / 'rank_05_estados.csv', index=False, encoding='utf-8-sig')
rank_regions.to_csv(ANALYSIS_DIR / 'rank_06_regioes.csv', index=False, encoding='utf-8-sig')
rank_years.to_csv(ANALYSIS_DIR / 'rank_07_anos.csv', index=False, encoding='utf-8-sig')
rank_fields_v3.to_csv(ANALYSIS_DIR / 'rank_08_campos_estudo_v3.csv', index=False, encoding='utf-8-sig')
rank_fields_main_v3.to_csv(ANALYSIS_DIR / 'rank_08b_campo_principal_v3.csv', index=False, encoding='utf-8-sig')
rank_methodologies.to_csv(ANALYSIS_DIR / 'rank_09_metodologias.csv', index=False, encoding='utf-8-sig')
rank_cited_authors.to_csv(ANALYSIS_DIR / 'rank_10_autores_citados_v1.csv', index=False, encoding='utf-8-sig')
rank_adherence.to_csv(ANALYSIS_DIR / 'rank_11_aderencia_elites.csv', index=False, encoding='utf-8-sig')

# Versões amigáveis para Sheets dos principais rankings
rank_authors.rename(columns={'author': 'autor', 'productions': 'producoes'}).to_csv(ANALYSIS_DIR / 'sheets_rank_01_autores.csv', index=False, encoding='utf-8-sig')
rank_journals.rename(columns={'journal': 'periodico', 'productions': 'producoes', 'first_year': 'primeiro_ano', 'last_year': 'ultimo_ano'}).to_csv(ANALYSIS_DIR / 'sheets_rank_02_periodicos.csv', index=False, encoding='utf-8-sig')
rank_keywords.rename(columns={'keyword': 'palavra_chave', 'productions': 'producoes'}).to_csv(ANALYSIS_DIR / 'sheets_rank_03_palavras_chave.csv', index=False, encoding='utf-8-sig')
rank_institutions.rename(columns={'institution': 'instituicao', 'productions': 'producoes'}).to_csv(ANALYSIS_DIR / 'sheets_rank_04_instituicoes.csv', index=False, encoding='utf-8-sig')
rank_fields_v3.rename(columns={'field': 'campo_estudo', 'productions': 'producoes'}).to_csv(ANALYSIS_DIR / 'sheets_rank_08_campos_estudo_v3.csv', index=False, encoding='utf-8-sig')
rank_fields_main_v3.rename(columns={'campo_principal_v3': 'campo_principal', 'productions': 'producoes'}).to_csv(ANALYSIS_DIR / 'sheets_rank_08b_campo_principal_v3.csv', index=False, encoding='utf-8-sig')
rank_methodologies.rename(columns={'methodology': 'metodologia', 'productions': 'producoes'}).to_csv(ANALYSIS_DIR / 'sheets_rank_09_metodologias.csv', index=False, encoding='utf-8-sig')
rank_adherence.rename(columns={'aderencia_elites_v1': 'aderencia_elites', 'productions': 'producoes'}).to_csv(ANALYSIS_DIR / 'sheets_rank_11_aderencia_elites.csv', index=False, encoding='utf-8-sig')

# Cruzamentos
field_year.to_csv(ANALYSIS_DIR / 'cross_01_campo_por_ano.csv', index=False, encoding='utf-8-sig')
field_main_year.to_csv(ANALYSIS_DIR / 'cross_01b_campo_principal_por_ano.csv', index=False, encoding='utf-8-sig')
method_year.to_csv(ANALYSIS_DIR / 'cross_02_metodologia_por_ano.csv', index=False, encoding='utf-8-sig')
region_year.to_csv(ANALYSIS_DIR / 'cross_03_regiao_por_ano.csv', index=False, encoding='utf-8-sig')
field_region.to_csv(ANALYSIS_DIR / 'cross_04_campo_principal_por_regiao.csv', index=False, encoding='utf-8-sig')
adherence_by_field.to_csv(ANALYSIS_DIR / 'cross_05_aderencia_por_campo.csv', index=False, encoding='utf-8-sig')
adherence_by_keyword.to_csv(ANALYSIS_DIR / 'cross_06_aderencia_por_keyword.csv', index=False, encoding='utf-8-sig')

# Excel consolidado
excel_path = ANALYSIS_DIR / 'analise_corpus_elites_scielo.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    diagnostic.to_excel(writer, sheet_name='diagnostico', index=False)
    corpus_unique_sheets.to_excel(writer, sheet_name='corpus_sheets', index=False)
    manual_validation_sample.to_excel(writer, sheet_name='validacao_5pct', index=False)
    rank_authors.to_excel(writer, sheet_name='autores', index=False)
    rank_journals.to_excel(writer, sheet_name='periodicos', index=False)
    rank_keywords.to_excel(writer, sheet_name='palavras_chave', index=False)
    rank_institutions.to_excel(writer, sheet_name='instituicoes', index=False)
    rank_states.to_excel(writer, sheet_name='estados', index=False)
    rank_regions.to_excel(writer, sheet_name='regioes', index=False)
    rank_years.to_excel(writer, sheet_name='anos', index=False)
    rank_fields_main_v3.to_excel(writer, sheet_name='campo_principal_v3', index=False)
    rank_fields_v3.to_excel(writer, sheet_name='campos_v3', index=False)
    rank_methodologies.to_excel(writer, sheet_name='metodologias', index=False)
    rank_cited_authors.to_excel(writer, sheet_name='autores_citados_v1', index=False)
    rank_adherence.to_excel(writer, sheet_name='aderencia_elites', index=False)
    field_main_year.to_excel(writer, sheet_name='campo_principal_por_ano', index=False)
    field_year.to_excel(writer, sheet_name='campos_por_ano', index=False)
    method_year.to_excel(writer, sheet_name='metodo_por_ano', index=False)
    region_year.to_excel(writer, sheet_name='regiao_por_ano', index=False)
    field_region.to_excel(writer, sheet_name='campo_por_regiao', index=False)
    adherence_by_field.to_excel(writer, sheet_name='aderencia_por_campo', index=False)
    adherence_by_keyword.to_excel(writer, sheet_name='aderencia_por_keyword', index=False)

print('Outputs salvos em:', ANALYSIS_DIR)
print('Excel:', excel_path)

## 16. Checagens rápidas

In [ ]:
display(diagnostic)
display(rank_fields_main_v3.head(20))
display(rank_adherence)